# Land cover classification at the Mississppi Delta

In this notebook, you will use a k-means **unsupervised** clustering
algorithm to group pixels by similar spectral signatures. **k-means** is
an **exploratory** method for finding patterns in data. Because it is
unsupervised, you don’t need any training data for the model. You also
can’t measure how well it “performs” because the clusters will not
correspond to any particular land cover class. However, we expect at
least some of the clusters to be identifiable as different types of land
cover.

You will use the [harmonized Sentinel/Landsat multispectral
dataset](https://lpdaac.usgs.gov/documents/1698/HLS_User_Guide_V2.pdf).
You can access the data with an [Earthdata
account](https://www.earthdata.nasa.gov/learn/get-started) and the
[`earthaccess` library from
NSIDC](https://github.com/nsidc/earthaccess):

## STEP 1: Set up

### Step 1a: Load libraries and set GDAL parameters

<link rel="stylesheet" type="text/css" href="./assets/styles.css"><div class="callout callout-style-default callout-titled callout-task"><div class="callout-header"><div class="callout-icon-container"><i class="callout-icon"></i></div><div class="callout-title-container flex-fill">Try It</div></div><div class="callout-body-container callout-body"><ol type="1">
<li>Import all libraries you will need for this analysis</li>
<li>Configure GDAL parameters to help avoid connection errors:
<code>python      os.environ["GDAL_HTTP_MAX_RETRY"] = "5"      os.environ["GDAL_HTTP_RETRY_DELAY"] = "1"</code></li>
</ol></div></div>

In [1]:
# path vars
import os

# serializing (save objects to disk)
import pickle

# regex
import re

# vs code warnings that are actually helpful
import warnings

# crs projections
import cartopy.crs as ccrs

# nasa api for HLS data access
import earthaccess

# spatial data analysis
import earthpy as et

# geodataframes
import geopandas as gpd

# visualization
import geoviews as gv
import hvplot.pandas
import hvplot.xarray

# arrays
import numpy as np

# dataframes
import pandas as pd

# rasters
import rioxarray as rxr
import rioxarray.merge as rxrmerge
import xarray as xr

# progress bar
from tqdm.notebook import tqdm
from ipywidgets import IntProgress
from IPython.display import display

# polygons
from shapely.geometry import Polygon

# k means clustering
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score ## performance metric

### set GDAL parameters
os.environ["GDAL_HTTP_MAX_RETRY"] = "5"
os.environ["GDAL_HTTP_RETRY_DELAY"] = "1"

### don't show non-critical warnings
warnings.simplefilter('ignore')

c:\Users\naho5798\AppData\Local\miniconda3\envs\earth-analytics-python\Lib\site-packages\earthpy\__init__.py:7: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_string


### Step 1b: Run the caching decorator

Below you can find code for a caching **decorator** which you can use in
your code. To use the decorator:

``` python
@cached(key, override)
def do_something(*args, **kwargs):
    ...
    return item_to_cache
```

This decorator will **pickle** the results of running the
`do_something()` function, and only run the code if the results don’t
already exist. To override the caching, for example temporarily after
making changes to your code, set `override=True`. Note that to use the
caching decorator, you must write your own function to perform each
task!

You might notice that typically in these assignments, we start by creating a data_dir to store our data files. Here, our caching decorator is making the data directory for us.

In [2]:
### make the caching decorator
def cached(func_key, override=False):
    """
    A decorator to cache function results
    
    Parameters
    ==========
    key: str
      File basename used to save pickled results
    override: bool
      When True, re-compute even if the results are already stored
    """
    def compute_and_cache_decorator(compute_function):
        """
        Wrap the caching function
        
        Parameters
        ==========
        compute_function: function
          The function to run and cache results
        """
        def compute_and_cache(*args, **kwargs):
            """
            Perform a computation and cache, or load cached result.
            
            Parameters
            ==========
            args
              Positional arguments for the compute function
            kwargs
              Keyword arguments for the compute function
            """
            ### Add an identifier from the particular function call
            if 'cache_key' in kwargs:
                key = '_'.join((func_key, kwargs['cache_key']))
            else:
                key = func_key

            ### define a file path based on the directory structure in earthpy
            path = os.path.join(
                
                ### earthpy directory
                et.io.HOME, 
                
                ### earthpy dataset
                et.io.DATA_NAME, 
                
                ### make a subdirectory called "jars"
                'jars', 
                
                ### use f-string (formatted string) to create a string by embedding the value
                ### of the variable "key" into the string 
                ### use .pickle file extension (a pickle file is a serialized python objecT)
                f'{key}.pickle')
            
            ### Check if the cache exists already or if we should override caching
            if not os.path.exists(path) or override:
                
                ### Make jars directory if needed
                os.makedirs(os.path.dirname(path), exist_ok=True)
                
                ### Run the compute function as the user did
                result = compute_function(*args, **kwargs)
                
                ### Pickle the object (save to file)
                ### open the file at filename
                with open(path, 'wb') as file:
                    
                    ### save the result without needing to recompute when loading
                    ### it back into Python
                    pickle.dump(result, file)
            
            ### if the file already exists/we are not overriding the cache
            else:
               
                ### Unpickle the object (load the cached result)
                with open(path, 'rb') as file:
                    
                    ### use pickle.load to unserialize the file back into a python object
                    result = pickle.load(file)
                    
            return result
        
        return compute_and_cache
    
    return compute_and_cache_decorator

## STEP 2: Study site

For this analysis, you will use a watershed from the [Water Boundary
Dataset](https://www.usgs.gov/national-hydrography/access-national-hydrography-products),
HU12 watersheds (WBDHU12.shp).

<link rel="stylesheet" type="text/css" href="./assets/styles.css"><div class="callout callout-style-default callout-titled callout-task"><div class="callout-header"><div class="callout-icon-container"><i class="callout-icon"></i></div><div class="callout-title-container flex-fill">Try It</div></div><div class="callout-body-container callout-body"><ol type="1">
<li>Download the Water Boundary Dataset for region 8 (Mississippi)</li>
<li>Select watershed 080902030506</li>
<li>Generate a site map of the watershed</li>
</ol>
<p>Try to use the <strong>caching decorator</strong></p></div></div>

We chose this watershed because it covers parts of New Orleans an is
near the Mississippi Delta. Deltas are boundary areas between the land
and the ocean, and as a result tend to contain a rich variety of
different land cover and land use types.

In [3]:
### assign the hydrologic unit code
HUC_LEVEL=12

# download, unzip and read shapefile
@cached(f'wbd_08_hu{HUC_LEVEL}_gdf')

# function for reading in the file
def read_wbd_file(wbd_filename, cache_key):

    # url
    wbd_url = (

        # url for pulling huc 12 shapefile
        "https://prd-tnm.s3.amazonaws.com/StagedProducts/Hydrography/WBD/HU2/Shape/"

        # specific file name
        f'{wbd_filename}.zip'
    )

    # download data and unzip to a directory
    wbd_dir = et.data.get_data(url = wbd_url)

    # path to shapefil in directory
    wbd_path = os.path.join(wbd_dir,
                            'Shape',
                            f'WBDHU{HUC_LEVEL}.shp')
    
    # read shp and a gdf
    wbd_gdf = gpd.read_file(wbd_path,
                            
                            # pyogrio library
                            engine = 'pyogrio')
    
    # gdf for watershed boundaries
    return wbd_gdf


In [4]:
### open the shapefile using the read_wbd_file function that we created
wbd_gdf = read_wbd_file("WBD_08_HU2_Shape",
                        f'hu{HUC_LEVEL}')

In [5]:
wbd_gdf.head(5)

,tnmid,metasource,sourcedata,sourceorig,sourcefeat,loaddate,referenceg,areaacres,areasqkm,states,...,name,hutype,humod,tohuc,noncontrib,noncontr_1,shape_Leng,shape_Area,ObjectID,geometry
0,{8AFB1AF9-7296-4303-89DE-14CD073B859A},{511D2AC8-11BA-45FC-AB98-F69D693D4C44},Watershed Boundary Dataset (WBD),Natural Resources and Conservation Service and...,None,2024-08-15,"535297,540579",29441.81,119.15,LA,...,Gourd Bayou-Youngs Bayou,S,"LE,ID,DD",080500011308,0.0,0.0,NaN,NaN,1,"POLYGON ((-92.00021 32.53586, -91.99994 32.535..."
1,{916A17A6-B4A0-4FD7-9BB8-FFD1936B15B2},{511D2AC8-11BA-45FC-AB98-F69D693D4C44},Watershed Boundary Dataset (WBD),Natural Resources and Conservation Service and...,None,2024-08-15,535512,11406.67,46.16,LA,...,Hams Creek,S,ID,080802050104,0.0,0.0,NaN,NaN,2,"POLYGON ((-93.37574 30.58982, -93.3747 30.5891..."
2,{493C7EC1-2F1C-4B84-AFFB-6F6868A9868E},{511D2AC8-11BA-45FC-AB98-F69D693D4C44},Watershed Boundary Dataset (WBD),Natural Resources and Conservation Service and...,None,2024-08-15,"547190,559640",29138.21,117.92,LA,...,Caney Creek-Bayou D'Arbonne,S,NM,080402060503,0.0,0.0,NaN,NaN,3,"POLYGON ((-93.07761 32.88752, -93.07784 32.887..."
3,{49A3C087-B460-4F97-9D99-78CBB675248B},{511D2AC8-11BA-45FC-AB98-F69D693D4C44},Watershed Boundary Dataset (WBD),Natural Resources and Conservation Service and...,None,2024-08-15,"77417,78285",17759.39,71.87,AR,...,L'Aigle Creek-Saline River,S,NM,080402020206,0.0,0.0,NaN,NaN,4,"POLYGON ((-92.08947 33.29383, -92.0897 33.2938..."
4,{0FB41498-11EA-4AB1-AF05-E2A8E5E2E274},{511D2AC8-11BA-45FC-AB98-F69D693D4C44},Watershed Boundary Dataset (WBD),Natural Resources and Conservation Service and...,None,2024-08-15,1628466,98564.62,398.88,LA,...,West Cote Blanche Bay,W,NM,080801030800,0.0,0.0,NaN,NaN,5,"POLYGON ((-91.62408 29.73947, -91.62195 29.737..."


In [6]:
### filter the shapefile to the specific watershed we're using

### define the gdf for the watershed by subsetting the gdf of the whole watershed dataset
delta_gdf = wbd_gdf[wbd_gdf[

    ### filter the gdf to the row(s) with the watershe we want
    # dissolve to merge multipolygons
    f'huc{HUC_LEVEL}'].isin(['080902030506'])].dissolve()


### check it out
delta_gdf

,geometry,tnmid,metasource,sourcedata,sourceorig,sourcefeat,loaddate,referenceg,areaacres,areasqkm,...,huc12,name,hutype,humod,tohuc,noncontrib,noncontr_1,shape_Leng,shape_Area,ObjectID
0,"POLYGON ((-89.97047 29.74687, -89.96593 29.750...",{E942B72E-599E-48F5-908A-EA5265701C14},{511D2AC8-11BA-45FC-AB98-F69D693D4C44},Watershed Boundary Dataset (WBD),Natural Resources and Conservation Service and...,None,2024-08-15,"536881,539539",37355.86,151.17,...,080902030506,Manuel Canal-Spanish Lake,D,GC,080902030508,0.0,0.0,NaN,NaN,2560


In [7]:
### Make a site map with satellite imagery in the background
(
    # match delta_gdf projection to esri imagery (mercator)
    delta_gdf.to_crs(ccrs.Mercator())

    # hv plot
    .hvplot(

        # watershed slightly transparent
        alpha = 0.35, fill_color = "white",

        # add sat imagery
        tiles = "EsriImagery",

        # plot in mercator
        crs = ccrs.Mercator())

    # set plot size
    .opts(width = 600, height = 500)

    
    )

:Overlay
   .WMTS.I     :WMTS   [Longitude,Latitude]
   .Polygons.I :Polygons   [Longitude,Latitude]

<link rel="stylesheet" type="text/css" href="./assets/styles.css"><div class="callout callout-style-default callout-response"><div class="callout-header"><div class="callout-icon-container"><i class="callout-icon"></i></div></div><div class="callout-body-container callout-body"><p>Write a 2-3 sentence <strong>site description</strong> (with
citations) of this area that helps to put your analysis in context.</p></div></div>


**SITE DESCRIPTION**
The HUC12 watershed, Manuel Canal–Spanish Lake (080902030506) is nested within the broader HUC2 Lower Mississippi Region watershed. Manuel Canal_Spanish Lake is southeast of New Orleans, LA. This area is characterized by water, wetlands, and some low upland ridges ([Day et al., 2007](https://www.science.org/doi/abs/10.1126/science.1137030?casa_token=8qqvl73qt8EAAAAA:nw9f2I21ih5y5DKEaxK_P-ArBFFR_YKMf7RMZQsoKkjCqjkQDWMkuKy-TrJQC9X9PArSV3aCd_GbB9s)), with the amount of inundated area chaging year to year ([Sentinel 2 Land Cover Explorer](https://livingatlas.arcgis.com/landcoverexplorer/#mapCenter=-89.86717%2C29.74925%2C11.95&mode=step&timeExtent=2017%2C2024&year=2024&showImageryLayer=true&renderingRule=0)). The region is experiencing wide-spread wetland loss due to sea level rise, reduced sediement transport from the Mississippi River, and canal construction ([Day et al., 2007](https://www.science.org/doi/abs/10.1126/science.1137030?casa_token=8qqvl73qt8EAAAAA:nw9f2I21ih5y5DKEaxK_P-ArBFFR_YKMf7RMZQsoKkjCqjkQDWMkuKy-TrJQC9X9PArSV3aCd_GbB9s)).

## STEP 3: Multispectral data

### Step 3a: Search for data

<link rel="stylesheet" type="text/css" href="./assets/styles.css"><div class="callout callout-style-default callout-titled callout-task"><div class="callout-header"><div class="callout-icon-container"><i class="callout-icon"></i></div><div class="callout-title-container flex-fill">Try It</div></div><div class="callout-body-container callout-body"><ol type="1">
<li>Log in to the <code>earthaccess</code> service using your Earthdata
credentials:
<code>python      earthaccess.login(persist=True)</code></li>
<li>Modify the following sample code to search for granules of the
HLSL30 product overlapping the watershed boundary from May to October
2023 (there should be 76 granules):
<code>python      results = earthaccess.search_data(          short_name="...",          cloud_hosted=True,          bounding_box=tuple(gdf.total_bounds),          temporal=("...", "..."),      )</code></li>
</ol></div></div>

In [8]:
### Log in to earthaccess
earthaccess.login(persist=True)

In [9]:
### Search for HLS granules we want
results = earthaccess.search_data(

    ### specify which dataset and spatial resolution we want 
    short_name = "HLSL30",

    ### specify that we're using cloud data
    cloud_hosted = True,

    ### use the bounding box from our watershed boundary
    bounding_box=tuple(delta_gdf.total_bounds),

    ### set the temporal range of the data
    temporal = ("2024-06","2024-08")
)

In [10]:
results

[Collection: {'EntryTitle': 'HLS Landsat Operational Land Imager Surface Reflectance and TOA Brightness Daily Global 30m v2.0'}
 Spatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'GPolygons': [{'Boundary': {'Points': [{'Longitude': -89.79864173, 'Latitude': 29.70347853}, {'Longitude': -89.76643746, 'Latitude': 30.69278312}, {'Longitude': -90.91181412, 'Latitude': 30.71627038}, {'Longitude': -90.93262544, 'Latitude': 29.72659663}, {'Longitude': -89.79864173, 'Latitude': 29.70347853}]}}]}}}
 Temporal coverage: {'RangeDateTime': {'BeginningDateTime': '2024-06-07T16:31:11.509Z', 'EndingDateTime': '2024-06-07T16:31:11.509Z'}}
 Size(MB): 169.50417041778564
 Data: ['https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/HLSL30.020/HLS.L30.T15RYP.2024159T163111.v2.0/HLS.L30.T15RYP.2024159T163111.v2.0.B10.tif', 'https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/HLSL30.020/HLS.L30.T15RYP.2024159T163111.v2.0/HLS.L30.T15RYP.2024159T163111.v2.0.SAA.tif', 'https://data.l

In [11]:
# Look at one of the granules
granule = results[0]
granule

Collection: {'EntryTitle': 'HLS Landsat Operational Land Imager Surface Reflectance and TOA Brightness Daily Global 30m v2.0'}
Spatial coverage: {'HorizontalSpatialDomain': {'Geometry': {'GPolygons': [{'Boundary': {'Points': [{'Longitude': -89.79864173, 'Latitude': 29.70347853}, {'Longitude': -89.76643746, 'Latitude': 30.69278312}, {'Longitude': -90.91181412, 'Latitude': 30.71627038}, {'Longitude': -90.93262544, 'Latitude': 29.72659663}, {'Longitude': -89.79864173, 'Latitude': 29.70347853}]}}]}}}
Temporal coverage: {'RangeDateTime': {'BeginningDateTime': '2024-06-07T16:31:11.509Z', 'EndingDateTime': '2024-06-07T16:31:11.509Z'}}
Size(MB): 169.50417041778564
Data: ['https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/HLSL30.020/HLS.L30.T15RYP.2024159T163111.v2.0/HLS.L30.T15RYP.2024159T163111.v2.0.B10.tif', 'https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/HLSL30.020/HLS.L30.T15RYP.2024159T163111.v2.0/HLS.L30.T15RYP.2024159T163111.v2.0.SAA.tif', 'https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/HLSL30.020/HLS.L30.T15RYP.2024159T163111.v2.0/HLS.L30.T15RYP.2024159T163111.v2.0.VZA.tif', 'https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/HLSL30.020/HLS.L30.T15RYP.2024159T163111.v2.0/HLS.L30.T15RYP.2024159T163111.v2.0.B06.tif', 'https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/HLSL30.020/HLS.L30.T15RYP.2024159T163111.v2.0/HLS.L30.T15RYP.2024159T163111.v2.0.B09.tif', 'https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/HLSL30.020/HLS.L30.T15RYP.2024159T163111.v2.0/HLS.L30.T15RYP.2024159T163111.v2.0.B04.tif', 'https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/HLSL30.020/HLS.L30.T15RYP.2024159T163111.v2.0/HLS.L30.T15RYP.2024159T163111.v2.0.B03.tif', 'https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/HLSL30.020/HLS.L30.T15RYP.2024159T163111.v2.0/HLS.L30.T15RYP.2024159T163111.v2.0.B07.tif', 'https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/HLSL30.020/HLS.L30.T15RYP.2024159T163111.v2.0/HLS.L30.T15RYP.2024159T163111.v2.0.VAA.tif', 'https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/HLSL30.020/HLS.L30.T15RYP.2024159T163111.v2.0/HLS.L30.T15RYP.2024159T163111.v2.0.Fmask.tif', 'https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/HLSL30.020/HLS.L30.T15RYP.2024159T163111.v2.0/HLS.L30.T15RYP.2024159T163111.v2.0.B01.tif', 'https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/HLSL30.020/HLS.L30.T15RYP.2024159T163111.v2.0/HLS.L30.T15RYP.2024159T163111.v2.0.SZA.tif', 'https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/HLSL30.020/HLS.L30.T15RYP.2024159T163111.v2.0/HLS.L30.T15RYP.2024159T163111.v2.0.B11.tif', 'https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/HLSL30.020/HLS.L30.T15RYP.2024159T163111.v2.0/HLS.L30.T15RYP.2024159T163111.v2.0.B05.tif', 'https://data.lpdaac.earthdatacloud.nasa.gov/lp-prod-protected/HLSL30.020/HLS.L30.T15RYP.2024159T163111.v2.0/HLS.L30.T15RYP.2024159T163111.v2.0.B02.tif']

### Step 3b: Compile information about each granule

I recommend building a GeoDataFrame, as this will allow you to plot the
granules you are downloading and make sure they line up with your
shapefile. You could also use a DataFrame, dictionary, or a custom
object to store this information.

<link rel="stylesheet" type="text/css" href="./assets/styles.css"><div class="callout callout-style-default callout-titled callout-task"><div class="callout-header"><div class="callout-icon-container"><i class="callout-icon"></i></div><div class="callout-title-container flex-fill">Try It</div></div><div class="callout-body-container callout-body"><ol type="1">
<li>For each search result:
<ol type="1">
<li>Get the following information (HINT: look at the [‘umm’] values for
each search result):
<ul>
<li>granule id (UR)</li>
<li>datetime</li>
<li>geometry (HINT: check out the shapely.geometry.Polygon class to
convert points to a Polygon)</li>
</ul></li>
<li>Open the granule files. I recommend opening one granule at a time,
e.g. with (<code>earthaccess.open([result]</code>).</li>
<li>For each file (band), get the following information:
<ul>
<li>file handler returned from <code>earthaccess.open()</code></li>
<li>tile id</li>
<li>band number</li>
</ul></li>
</ol></li>
<li>Compile all the information you collected into a GeoDataFrame</li>
</ol></div></div>

In [12]:
### make a function to process all the granules from the earthaccess search
### and extract information for each granule

### define the function
def get_earthaccess_links(results):

    ### make and display a progress bar
    f = IntProgress(min = 0, max = len(results), description = 'Open granules')
    display(f)

    ### use a regular expression to extract tile_id and bank from .tif files
    url_re = re.compile(
        r'\.(?P<tile_id>\w+)\.\d+T\d+\.v\d\.\d\.(?P<band>[A-Za-z0-9]+)\.tif')

    ### accumulate gdf rows from each granule
    link_rows = []    

    ### loop over granules to extract info
    for granule in results:

        ### locate metadata (UMM = universal metadata model)
        info_dict = granule['umm']

        ### pull out unique identifier for the granule
        granule_id = info_dict['GranuleUR']

        ### extract date/time 
        datetime = pd.to_datetime(
            info_dict['TemporalExtent']['RangeDateTime']['BeginningDateTime']
        )

        ### extact boundary coordinates for granule
        points = (
            info_dict
            ['SpatialExtent']['HorizontalSpatialDomain']['Geometry']['GPolygons'][0]
            ['Boundary']['Points']
        )

        ### make polygon using coordinate points for granule
        geometry = Polygon(
            [(point['Longitude'],
              point['Latitude']) for point in points]
        )

        
        ### get url and open granule
        files = earthaccess.open([granule])

        ### loop through each file in the granule
        for file in files:

            ### use url regular expression to get url
            match = url_re.search(file.full_name)

            ### if match is found, append data to link_rows gdf we initialized
            if match is not None:
                link_rows.append(

                    ### makes a gdf with the granule's data and geometry
                    gpd.GeoDataFrame(
                        dict(

                            # timestamp
                            datetime = [datetime],
                            
                            # unique id
                            tile_id = [match.group('tile_id')],
                            
                            # band name
                            band = [match.group('band')],

                            # url
                            url = [file],

                            # polygon
                            geometry = [geometry]
                        ),

                        # set crs
                        crs = "EPSG:4326"
                    )
                )
         

        ### update progress bar after each granule is done
        f.value += 1

    ### combine into a single gdf   
    file_df = pd.concat(link_rows).reset_index(drop = True)

    ### return the final gdf file
    return file_df

In [13]:
# run the function to get granules
file_df = get_earthaccess_links(results)

IntProgress(value=0, description='Open granules', max=44)

QUEUEING TASKS | :   0%|          | 0/15 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/15 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/15 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/15 [00:00<?, ?it/s]

...

PROCESSING TASKS | :   0%|          | 0/15 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/15 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/15 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/15 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/15 [00:00<?, ?it/s]

In [14]:
type(file_df)

geopandas.geodataframe.GeoDataFrame

In [15]:
file_df.shape[0]

660

### Step 3c: Open, crop, and mask data

This will be the most resource-intensive step. I recommend caching your
results using the `cached` decorator or by writing your own caching
code. I also recommend testing this step with one or two dates before
running the full computation.

This code should include at least one **function** including a
numpy-style docstring. A good place to start would be a function for
opening a single masked raster, applying the appropriate scale
parameter, and cropping.

<link rel="stylesheet" type="text/css" href="./assets/styles.css"><div class="callout callout-style-default callout-titled callout-task"><div class="callout-header"><div class="callout-icon-container"><i class="callout-icon"></i></div><div class="callout-title-container flex-fill">Try It</div></div><div class="callout-body-container callout-body"><ol type="1">
<li>For each granule:
<ol type="1">
<li><p>Open the Fmask band, crop, and compute a quality mask for the
granule. You can use the following code as a starting point, making sure
that <code>mask_bits</code> contains the quality bits you want to
consider: ```python # Expand into a new dimension of binary bits bits =
( np.unpackbits(da.astype(np.uint8), bitorder=‘little’)
.reshape(da.shape + (-1,)) )</p>
<p># Select the required bits and check if any are flagged mask =
np.prod(bits[…, mask_bits]==0, axis=-1) ```</p></li>
<li><p>For each band that starts with ‘B’:</p>
<ol type="1">
<li>Open the band, crop, and apply the scale factor</li>
<li>Name the DataArray after the band using the <code>.name</code>
attribute</li>
<li>Apply the cloud mask using the <code>.where()</code> method</li>
<li>Store the DataArray in your data structure (e.g. adding a
GeoDataFrame column with the DataArray in it. Note that you will need to
remove the rows for unused bands)</li>
</ol></li>
</ol></li>
</ol></div></div>

In [16]:
### apply cached decorator to function
@cached('delta_reflectance_da_df')


### write function that computes reflectance data using 
### search results (df of urls) and watershed boundary
def compute_reflectance_da(search_results, boundary_gdf):

    """
    Connect to files using VSI, crop then, apply a cloud mask, and wrangle

    Return a single refelctance DataFrame with bands as columns
    and centroid coordinates and datetime as the index

    Parameters
    ==========
    search_results:list
        search result links to the files (urls)
    boundary_gdf: gpd.GeoDataFrame
        boundary used to crop the data
    """
    ### write a function to open raster from url, apply scale factor, crop, and mask data
    def open_dataarray(url, boundary_proj_gdf, scale = 1, masked = True):
        """
        Open raster data, convert crs if none, and crop to boundary

        Parameters
        ==========
        input here
         """
    
        # open raster data
        da = rxr.open_rasterio(url, masked = masked).squeeze() * scale
        
        # reproject the boundary to match the raster crs
        if boundary_proj_gdf is None:
            boundary_proj_gdf = boundary_gdf.to_crs(da.rio.crs)

        # crop raster to bounding box
        cropped = da.rio.clip_box(*boundary_proj_gdf.total_bounds)

        return cropped

    ### write function to apply a cloud mask
    def compute_quality_mask(da, mask_bits = [1, 2, 3]):
        """
        Mask low quality data

        Parameters
        ==========
        input here
         """
        
        # unpack bits to axis
        bits = (

            # unpack each number into bits
            np.unpackbits(

                # convert to 8-bit integer format
                da.astype(np.uint8),

                # set order of bits
                bitorder = "little"
            
            # reshape to match original data + bit dimension
            ).reshape(da.shape + (-1, ))
        )

        # grab specific band bit flags
        mask = np.prod(

            bits[
                ...,
                            mask_bits] == 0,
                            axis = -1)

        ### return the mask
        return mask

    ### grab metadata
    file_df = get_earthaccess_links(search_results)

    # store results for each granule
    granule_da_rows = []

    # store project boundary
    boundary_proj_gdf = None

    # group data by each granule
    group_iter = file_df.groupby(

        # datatime and tile_id
        ['datetime', 'tile_id']
    )


    ### loop through each image and its metadata
    for (datetime, tile_id), granule_df in tqdm(group_iter):

        # status bar
        print(f'Processing granule {tile_id} {datetime}')

        # find each granule's cloud mask file (fmask) url
        cloud_mask_url = (
            granule_df.loc[granule_df.band == 'Fmask', 'url']
            .values[0])

        ### open granule cloud cover
        cloud_masked_cropped_da = open_dataarray(cloud_mask_url, boundary_proj_gdf, masked = False)

        ### compute cloud mask
        cloud_mask = compute_quality_mask(cloud_masked_cropped_da)


        ### loop through each spectral band to open, crop, and mask the band
        da_list = []
        df_list = []
        
        for i, row in granule_df.iterrows():

            # loop through spectral bands
            if row.band.startswith('B'):

                # open band raster and scale reflectance
                band_cropped = open_dataarray(
                    row.url, boundary_proj_gdf, scale = 0.0001
                )

                # name the raster by the band
                band_cropped.name = row.band

                # apply the cloud mask to the raster
                row['da'] = band_cropped.where(cloud_mask)
 
                ### append the row to granule_da_rows
                granule_da_rows.append(row.to_frame().T)


    ### reassemble the metadata df
    return pd.concat(granule_da_rows)



In [17]:
### apply the function
reflectance_da_df = compute_reflectance_da(results, delta_gdf)

In [18]:
### check out the dataframe
reflectance_da_df

,datetime,tile_id,band,url,geometry,da
1,2024-06-02 16:58:47.456000+00:00,T15TVH,B02,"<File-like object HTTPFileSystem, https://data...","POLYGON ((-94.21477274 42.35763631, -93.172205...",[[<xarray.DataArray 'B02' ()> Size: 4B\narray(...
4,2024-06-02 16:58:47.456000+00:00,T15TVH,B01,"<File-like object HTTPFileSystem, https://data...","POLYGON ((-94.21477274 42.35763631, -93.172205...",[[<xarray.DataArray 'B01' ()> Size: 4B\narray(...
5,2024-06-02 16:58:47.456000+00:00,T15TVH,B04,"<File-like object HTTPFileSystem, https://data...","POLYGON ((-94.21477274 42.35763631, -93.172205...",[[<xarray.DataArray 'B04' ()> Size: 4B\narray(...
7,2024-06-02 16:58:47.456000+00:00,T15TVH,B06,"<File-like object HTTPFileSystem, https://data...","POLYGON ((-94.21477274 42.35763631, -93.172205...",[[<xarray.DataArray 'B06' ()> Size: 4B\narray(...
8,2024-06-02 16:58:47.456000+00:00,T15TVH,B07,"<File-like object HTTPFileSystem, https://data...","POLYGON ((-94.21477274 42.35763631, -93.172205...",[[<xarray.DataArray 'B07' ()> Size: 4B\narray(...
...,...,...,...,...,...,...
280,2024-08-30 16:53:00.827000+00:00,T15TVH,B10,"<File-like object HTTPFileSystem, https://data...","POLYGON ((-93.94378469 42.36018743, -92.881472...",[[<xarray.DataArray 'B10' ()> Size: 4B\narray(...
281,2024-08-30 16:53:00.827000+00:00,T15TVH,B05,"<File-like object HTTPFileSystem, https://data...","POLYGON ((-93.94378469 42.36018743, -92.881472...",[[<xarray.DataArray 'B05' ()> Size: 4B\narray(...
282,2024-08-30 16:53:00.827000+00:00,T15TVH,B02,"<File-like object HTTPFileSystem, https://data...","POLYGON ((-93.94378469 42.36018743, -92.881472...",[[<xarray.DataArray 'B02' ()> Size: 4B\narray(...
283,2024-08-30 16:53:00.827000+00:00,T15TVH,B06,"<File-like object HTTPFileSystem, https://data...","POLYGON ((-93.94378469 42.36018743, -92.881472...",[[<xarray.DataArray 'B06' ()> Size: 4B\narray(...


### Step 3d: Merge and Composite Data

You will notice for this watershed that:   
1. The raster data for each date are spread across 4 granules  
2. Any given image is incomplete because of clouds

<link rel="stylesheet" type="text/css" href="./assets/styles.css"><div class="callout callout-style-default callout-titled callout-task"><div class="callout-header"><div class="callout-icon-container"><i class="callout-icon"></i></div><div class="callout-title-container flex-fill">Try It</div></div><div class="callout-body-container callout-body"><ol type="1">

*   1. For each band:  
    *   a. For each date:  
        *   i. Merge all 4 granules  
        *   ii. Mask any negative values created by interpolating from the nodata value of -9999 (`rioxarray`) should account for this, but doesn't appear to when merging. If you leave these values in, they will create problems later on
    *   b. Concatenate the merged DataArrays along a new date dimension  
    *   c. Take the mean in the date dimension to create a composite image that fills cloud gaps  
    *   d. Add the band as a dimensions, and give the DataArray a name  
*   2. Concatenate along the band dimension


In [19]:
### apply cache decorator
@cached('delta_reflectance_da')

### create a function to merge and composite reflectance data from multiple granules
### end result: single, composite reflectance image for each spectral band
def merge_and_composite_arrays(granule_da_df):

    ### initialize a list to store dfs
    da_list = []    

    ### loop over each spectral band
    for band, band_df in granule_da_df.groupby('band'):

        # list for storing marged data arrays
        merged_das = []

        ### loop over date/time of image acquisition and merge granules for each data
        for datetime, date_df in band_df.groupby('datetime'):

            # merge granules for each date
            merged_da = rxrmerge.merge_arrays(list(date_df.da))

            ### mask negative values (could be no data or invalid data)
            merged_da = merged_da.where(merged_da > 0)
            
            ### append to merged_das list we initialized
            merged_das.append(merged_da)
            
        ### composite images across dates
        composite_da = xr.concat(merged_das,
                                 
                                 # datetime dimension
                                 # median value across datetime for individual pixel
                                 dim = 'datetime').median('datetime')

        # assign band number to attribute of composite data array
        composite_da['band'] = int(band[1:])

        # name data comp data array
        composite_da.name = 'reflectance'

        ### add processed and composite data array to list
        da_list.append(composite_da)

    ### concatenates composite data arrays for each band along band dimension
    return xr.concat(da_list, dim = 'band')



In [20]:
### call function to get final composite reflectance data 
reflectance_da = merge_and_composite_arrays(reflectance_da_df)

In [21]:
reflectance_da

<xarray.DataArray 'reflectance' (band: 10, y: 399, x: 563)> Size: 9MB
array([[[0.0236    , 0.021     , 0.0237    , ..., 0.0165    ,
         0.01725   , 0.0173    ],
        [0.0243    , 0.0211    , 0.0228    , ..., 0.0182    ,
         0.01975   , 0.0198    ],
        [0.0218    , 0.0218    , 0.0217    , ..., 0.0181    ,
         0.01975   , 0.0194    ],
        ...,
        [0.0395    , 0.0272    , 0.0063    , ..., 0.048     ,
         0.0411    , 0.03365   ],
        [0.0309    , 0.0207    , 0.0081    , ..., 0.04925   ,
         0.0453    , 0.034     ],
        [0.01905   , 0.0141    , 0.01415   , ..., 0.05205   ,
         0.0489    , 0.03575   ]],

       [[0.0305    , 0.0262    , 0.0317    , ..., 0.0196    ,
         0.02      , 0.02      ],
        [0.0301    , 0.0259    , 0.029     , ..., 0.02095   ,
         0.02175   , 0.0223    ],
        [0.0277    , 0.027     , 0.0267    , ..., 0.02215   ,
         0.02255   , 0.0228    ],
...
        [0.232     , 0.22915   , 0.22755   , ..., 0.2408    ,
         0.23995   , 0.23705   ],
        [0.23075   , 0.22694999, 0.224     , ..., 0.24425   ,
         0.24305   , 0.23815   ],
        [0.2286    , 0.22415   , 0.22015   , ..., 0.24585   ,
         0.2445    , 0.23964998]],

       [[0.1987    , 0.19829999, 0.1986    , ..., 0.21675   ,
         0.21669999, 0.2085    ],
        [0.19999999, 0.1994    , 0.1998    , ..., 0.21689999,
         0.21675   , 0.20889999],
        [0.20099999, 0.20009999, 0.1995    , ..., 0.21714999,
         0.21689999, 0.2095    ],
        ...,
        [0.20589998, 0.20535   , 0.20425   , ..., 0.2183    ,
         0.2181    , 0.21805   ],
        [0.20475   , 0.20375   , 0.2022    , ..., 0.21845   ,
         0.21835   , 0.2183    ],
        [0.20324999, 0.20185   , 0.19985   , ..., 0.2186    ,
         0.2184    , 0.2182    ]]], shape=(10, 399, 563), dtype=float32)
Coordinates:
  * x            (x) float64 5kB 4.765e+05 4.766e+05 ... 4.934e+05 4.934e+05
  * y            (y) float64 3kB 4.721e+06 4.721e+06 ... 4.709e+06 4.709e+06
  * band         (band) int64 80B 1 2 3 4 5 6 7 9 10 11
    spatial_ref  int64 8B 0

## STEP 4: K-means clustering

Cluster your data by spectral signature using the k-means algorithm.

<link rel="stylesheet" type="text/css" href="./assets/styles.css"><div class="callout callout-style-default callout-titled callout-task"><div class="callout-header"><div class="callout-icon-container"><i class="callout-icon"></i></div><div class="callout-title-container flex-fill">Try It</div></div><div class="callout-body-container callout-body"><ol type="1">
<li>Convert your DataArray into a <strong>tidy</strong> DataFrame of
reflectance values (hint: check out the <code>.to_dataframe()</code> and
<code>.unstack()</code> methods)</li>
<li>Filter out all rows with no data (all 0s or any N/A values)</li>
<li>Fit a k-means model. You can experiment with the number of groups to
find what works best.</li>
</ol></div></div>

In [22]:
### Convert spectral DataArray to a tidy DataFrame
model_df = (reflectance_da
            
            # long dataframe
            .to_dataframe()

            # select reflectance column
            .reflectance

            # pivot wider (rows --> pixels, columns --> spectral bands)
            .unstack('band')
            )

model_df

### filter out rows with no data
model_df = model_df.drop(columns = [10,11]).dropna()
model_df

band                      1        2        3        4        5        6  \
y         x                                                                
4721265.0 476535.0  0.02360  0.03050  0.08600  0.03710  0.45510  0.24080   
          476565.0  0.02100  0.02620  0.08800  0.03730  0.46390  0.23090   
          476595.0  0.02370  0.03170  0.08260  0.04200  0.43280  0.21940   
          476625.0  0.02180  0.02660  0.06960  0.04210  0.37530  0.19380   
          476655.0  0.02180  0.02810  0.06470  0.04330  0.35740  0.19610   
...                     ...      ...      ...      ...      ...      ...   
4709325.0 493275.0  0.04945  0.05645  0.08730  0.08575  0.33450  0.33280   
          493305.0  0.05150  0.05570  0.09020  0.10030  0.35170  0.35690   
          493335.0  0.05205  0.05750  0.08815  0.08725  0.39580  0.34655   
          493365.0  0.04890  0.05345  0.08180  0.07750  0.36700  0.30655   
          493395.0  0.03575  0.03805  0.06810  0.04635  0.40275  0.21655   

band                      7        9  
y         x                           
4721265.0 476535.0  0.11380  0.00120  
          476565.0  0.11490  0.00140  
          476595.0  0.10610  0.00140  
          476625.0  0.09370  0.00150  
          476655.0  0.09130  0.00140  
...                     ...      ...  
4709325.0 493275.0  0.23560  0.00150  
          493305.0  0.25120  0.00140  
          493335.0  0.23535  0.00175  
          493365.0  0.20410  0.00155  
          493395.0  0.11620  0.00155  

[224636 rows x 8 columns]

Now we're reading to fit the k-means clustering model. We can run the fit and prediction functions at the same time because we don't have target data.

In [23]:
### initialize k-means model 
k_means = KMeans(n_clusters = 6)

### fit model and predict
prediction = k_means.fit_predict(model_df.values)

### add the predicted values back to the model dataframe
model_df['clusters'] = prediction

model_df

band                      1        2        3        4        5        6  \
y         x                                                                
4721265.0 476535.0  0.02360  0.03050  0.08600  0.03710  0.45510  0.24080   
          476565.0  0.02100  0.02620  0.08800  0.03730  0.46390  0.23090   
          476595.0  0.02370  0.03170  0.08260  0.04200  0.43280  0.21940   
          476625.0  0.02180  0.02660  0.06960  0.04210  0.37530  0.19380   
          476655.0  0.02180  0.02810  0.06470  0.04330  0.35740  0.19610   
...                     ...      ...      ...      ...      ...      ...   
4709325.0 493275.0  0.04945  0.05645  0.08730  0.08575  0.33450  0.33280   
          493305.0  0.05150  0.05570  0.09020  0.10030  0.35170  0.35690   
          493335.0  0.05205  0.05750  0.08815  0.08725  0.39580  0.34655   
          493365.0  0.04890  0.05345  0.08180  0.07750  0.36700  0.30655   
          493395.0  0.03575  0.03805  0.06810  0.04635  0.40275  0.21655   

band                      7        9  clusters  
y         x                                     
4721265.0 476535.0  0.11380  0.00120         2  
          476565.0  0.11490  0.00140         2  
          476595.0  0.10610  0.00140         2  
          476625.0  0.09370  0.00150         1  
          476655.0  0.09130  0.00140         1  
...                     ...      ...       ...  
4709325.0 493275.0  0.23560  0.00150         4  
          493305.0  0.25120  0.00140         0  
          493335.0  0.23535  0.00175         4  
          493365.0  0.20410  0.00155         4  
          493395.0  0.11620  0.00155         1  

[224636 rows x 9 columns]

## STEP 5: Plot

<link rel="stylesheet" type="text/css" href="./assets/styles.css"><div class="callout callout-style-default callout-titled callout-task"><div class="callout-header"><div class="callout-icon-container"><i class="callout-icon"></i></div><div class="callout-title-container flex-fill">Try It</div></div><div class="callout-body-container callout-body"><p>Create a plot that shows the k-means clusters next to an RGB image of
the area. You may need to brighten your RGB image by multiplying it by
10. The code for reshaping and plotting the clusters is provided for you
below, but you will have to create the RGB plot yourself!</p>
<p>So, what is <code>.sortby(['x', 'y'])</code> doing for us? Try the
code without it and find out.</p></div></div>

In [24]:
### make data array with bands to use for rgb: red, green, and blue
rgb = reflectance_da.sel(band = [4, 3, 2])

In [25]:
# plot rgb
(
    rgb.hvplot.rgb(y = 'y',
                   x = 'x',
                   bands = 'band',
                   data_aspect = 1,
                   xaxis = None,
                   yaxis = None)
)

:RGB   [x,y]   (R,G,B)

In [26]:
# strecth values so we can see the plot
rgb_uint8 = (rgb * 255).astype(np.uint8).where(rgb != np.nan)

In [27]:
# plot rgb scales
(
    rgb_uint8.hvplot.rgb(y = 'y',
                   x = 'x',
                   bands = 'band',
                   data_aspect = 1,
                   xaxis = None,
                   yaxis = None)
)

:RGB   [x,y]   (R,G,B)

In [28]:
# increase brightness
rgb_uint8_bright = rgb_uint8 * 10

In [29]:
# plot rgb scales
(
    rgb_uint8_bright.hvplot.rgb(y = 'y',
                   x = 'x',
                   bands = 'band',
                   data_aspect = 1,
                   xaxis = None,
                   yaxis = None)
)

:RGB   [x,y]   (R,G,B)

In [30]:
# cap saturation at 255
rgb_sat = rgb_uint8_bright.where(rgb_uint8_bright < 255, 255)

In [31]:
# plot rgb with saturation cap
(
    rgb_sat.hvplot.rgb(y = 'y',
                   x = 'x',
                   bands = 'band',
                   data_aspect = 1, # avoids stretching and distorting
                   xaxis = None,
                   yaxis = None)
)

:RGB   [x,y]   (R,G,B)

In [32]:
# plot the clusters from kmeans
(
    model_df.clusters.to_xarray().hvplot(y = 'y',
                   x = 'x',
                   bands = 'band',
                   data_aspect = 1, # avoids stretching and distorting
                   xaxis = None,
                   yaxis = None)
                   )

:Image   [x,y]   (clusters)

In [33]:
# deal with sorting issues of pixels
(
    model_df.clusters.to_xarray().sortby(['x', 'y']).hvplot(y = 'y',
                   x = 'x',
                   bands = 'band',
                   data_aspect = 1, # avoids stretching and distorting
                   xaxis = None,
                   yaxis = None)
                   )

:Image   [x,y]   (clusters)

In [34]:
### plot the k-means clusters

# convert the cluster values to integers before plotting to remove the decimal tick labels
#model_df["clusters"] = model_df["clusters"].astype('category')

# plot
delta_cluster_plot = (
    rgb_sat.hvplot.rgb(
                    y = 'y',
                    x = 'x',
                    bands = 'band',
                    data_aspect = 1, # avoids stretching and distorting
                    xaxis = None,
                    yaxis = None,
                    title="RGB imagery")
    + 
    model_df.clusters.to_xarray().sortby(['x', 'y']).hvplot(
        cmap="Colorblind", 
        aspect='equal',
        title="K-means clusters",
        xlabel="Easting",
        ylabel="Northing",
        colorbar=True) 
)

delta_cluster_plot

:Layout
   .RGB.I   :RGB   [x,y]   (R,G,B)
   .Image.I :Image   [x,y]   (clusters)

In [36]:
# save out the plot
hv.save(delta_cluster_plot, "delta_cluster_plot.html")


NameError: name 'hv' is not defined

#### Tried to get a silhouette run to test different clustering values but nothing would finish running!

In [ ]:
### Convert spectral DataArray to a tidy DataFrame
#model_df = (reflectance_da
#            
#            # long dataframe
#            .to_dataframe()
#
#            # select reflectance column
#            .reflectance
#
#            # pivot wider (rows --> pixels, columns --> spectral bands)
#            .unstack('band')
#            )

#model_df

### filter out rows with no data
#model_df = model_df.drop(columns = [10,11]).dropna()
#model_df

In [ ]:
### accummulate silhouette scores
#silhouette = []

### make a list of k values to loop through
#k_list = list(range(5, 6))

### loop through the k values
#for k in k_list:

    ### make model with k clusters
#    k_means = KMeans(n_clusters = k)

    ### ID the variables to include
    #model_vars = (penguins_df[['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']])

    ### fit model
#    k_means.fit(model_df)

    ### calculate silhouette score and add it the list we initialized, along with the corresponding k values
#    silhouette.append(silhouette_score(model_df, k_means.labels_))


### check it out
#silhouette

In [ ]:
### visualize 
sns.scatterplot(x = k_list, y = silhouette)

<link rel="stylesheet" type="text/css" href="./assets/styles.css"><div class="callout callout-style-default callout-titled callout-respond"><div class="callout-header"><div class="callout-icon-container"><i class="callout-icon"></i></div><div class="callout-title-container flex-fill">Reflect and Respond</div></div><div class="callout-body-container callout-body"><p>Don’t forget to interpret your plot!</p></div></div>

**K-means clustering struggles to identify accurate land cover classes**

The K-menas plot does not accurately identify distinct land cover classes. It is also difficult to use the RGB imagery to make out distinct land cover types, as most pixels have a green tint that makes distinguishing land and water a challenge. I instead used the [Sentinel-2 Land Cover Explorer](https://livingatlas.arcgis.com/landcoverexplorer/#mapCenter=-89.85118%2C29.74288%2C11.25&mode=step&timeExtent=2017%2C2024&showImageryLayer=true&renderingRule=0&month=9&year=2024) to assess the accuracy of the K-means clusters. While overall land cover classification does not appear to be accurate, there are a few glimmers of hope. The town of Delacroix (western edge of the boundary area, aka [End of the World](https://maps.app.goo.gl/EgqfFMrdDoVz5zWm6)) DOES appear to be distinctly clustered as a built environment (cluster 2, green). However, it also lumped this in with what appears to be marsh vegetation. Most of the open water pixels are split between cluster 1 (orange) and 4 (pink). 

It seems our clustering struggles with the transition pixels and marshland. The marshes of the Mississippi Deltaic Plain are zonal, with the vegetation communities in each zone relfecting salinity gradients from freshwater (e.g. *Sagittaria* spp., *Colocasia* spp., *Ludwigia* spp.), brackish (e.g. *Spartina patens*, *Typha*) and saline marsh (e.g. *Juncus* spp., *Spartina alterniflora*). See [Visser et al., 2012](https://www.researchgate.net/figure/Summary-of-the-different-habitats-in-the-Mississippi-Deltaic-Plain_tbl1_261645954) for a coarse map of these zones! These are distinctly different vegetation communities, usually accompanied by a difference in color and vegetation height you can see from a speeding car, so I was curious to see if the clustering could identify these communities. Unfortunately, it looks like we would need to tune this clustering more in order to better classify these vegetation communities.
